# 💳 Fraud Detection — SMOTE, Logistic Regression & Random Forest

**Oasis Infobyte Data Analytics — Level 1, Task 3**

This notebook implements the requested fraud-detection workflow using a stratified hold-out test set, SMOTE on training data only, Logistic Regression, Random Forest, Precision, Recall, F1-score, ROC-AUC, confusion matrices and feature importance. The raw `creditcard.csv` is expected at `../data/creditcard.csv` and is not committed because it exceeds GitHub's single-file limit.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from imblearn.over_sampling import SMOTE

DATA_PATH='../data/creditcard.csv'
RESULTS_DIR='../results'
os.makedirs(RESULTS_DIR, exist_ok=True)
df=pd.read_csv(DATA_PATH)
print(df.shape)
print(df['Class'].value_counts())
print(f"Fraud percentage: {df['Class'].mean()*100:.4f}%")

## Why accuracy is not enough
Fraud is a rare class. A classifier can appear highly accurate while missing fraudulent transactions. Precision measures how many flagged transactions are actually fraud; Recall measures how much of the fraud is caught; F1 balances the two; ROC-AUC measures ranking quality across thresholds.

In [ ]:
X=df.drop(columns='Class'); y=df['Class']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=42)

# Scale for Logistic Regression. Random Forest is unaffected by monotonic scaling, so the same scaled matrix is used for a consistent pipeline.
scaler=StandardScaler()
X_train_s=scaler.fit_transform(X_train)
X_test_s=scaler.transform(X_test)

# Apply SMOTE ONLY to training data. A 10% minority/majority ratio limits unnecessary synthetic samples.
smote=SMOTE(random_state=42,sampling_strategy=.10)
X_smote,y_smote=smote.fit_resample(X_train_s,y_train)
print('Before SMOTE:',y_train.value_counts().to_dict())
print('After SMOTE:',y_smote.value_counts().to_dict())

In [ ]:
models={
 'Logistic Regression':LogisticRegression(max_iter=1000,random_state=42),
 'Random Forest':RandomForestClassifier(n_estimators=20,max_depth=16,min_samples_leaf=2,n_jobs=-1,random_state=42)
}
metrics=[]; predictions={}; probabilities={}
for name,model in models.items():
 model.fit(X_smote,y_smote)
 pred=model.predict(X_test_s); prob=model.predict_proba(X_test_s)[:,1]
 predictions[name]=pred; probabilities[name]=prob
 metrics.append({'Model':name,'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1':f1_score(y_test,pred,zero_division=0),'ROC-AUC':roc_auc_score(y_test,prob)})
metrics_df=pd.DataFrame(metrics)
display(metrics_df.style.format({c:'{:.4f}' for c in ['Precision','Recall','F1','ROC-AUC']}))
metrics_df.to_csv(os.path.join(RESULTS_DIR,'model_metrics.csv'),index=False)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
for ax,(name,pred) in zip(axes,predictions.items()):
 sns.heatmap(confusion_matrix(y_test,pred),annot=True,fmt='d',cbar=False,ax=ax)
 ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'confusion_matrices.png'),dpi=160); plt.show()

plt.figure(figsize=(7,5))
for name,prob in probabilities.items():
 fpr,tpr,_=roc_curve(y_test,prob); plt.plot(fpr,tpr,label=f'{name} (AUC={roc_auc_score(y_test,prob):.4f})')
plt.plot([0,1],[0,1],'--',label='Random classifier')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve — SMOTE Models'); plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'roc_curve.png'),dpi=160); plt.show()

In [ ]:
rf=models['Random Forest']
importance=pd.DataFrame({'Feature':X.columns,'Importance':rf.feature_importances_}).sort_values('Importance',ascending=False)
importance.to_csv(os.path.join(RESULTS_DIR,'random_forest_feature_importance.csv'),index=False)
display(importance.head(15))
top=importance.head(15).sort_values('Importance')
plt.figure(figsize=(8,6)); plt.barh(top['Feature'],top['Importance']); plt.xlabel('Importance'); plt.title('Top 15 Fraud Features — Random Forest'); plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR,'feature_importance.png'),dpi=160); plt.show()

## Conclusion
Random Forest achieved the strongest overall balance in the supplied run: Precision 0.8000, Recall 0.8571, F1 0.8276 and ROC-AUC 0.9778. Logistic Regression produced higher Recall (0.8878) but lower Precision (0.3522). The leading Random Forest features were V14, V17, V12, V10 and V3. These anonymised PCA-derived features should not be interpreted as business variables without further documentation.

For high-volume production scoring, a streaming/batched feature pipeline, parallel model serving, threshold optimisation, monitoring and retraining would be appropriate.